In [37]:
import pandas as pd
import numpy as np

In [38]:
df = pd.read_csv("../data/raw/epiclim_raw.csv")
print("Shape:", df.shape)

Shape: (8985, 15)


In [39]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [40]:
df['Cases']  = pd.to_numeric(df['Cases'],  errors='coerce').fillna(0)
df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)

In [41]:
df.dropna(subset=['state_ut', 'district', 'Cases'], inplace=True)

In [42]:
for col in ['preci', 'LAI', 'Temp']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

print("After cleaning — shape:", df.shape)

After cleaning — shape: (8985, 14)


In [43]:
df['date'] = pd.to_datetime(
    df[['year', 'mon', 'day']].rename(columns={'year':'year','mon':'month','day':'day'})
)
df['year_month'] = df['date'].dt.to_period('M').dt.to_timestamp()
df['Month']      = df['date'].dt.month

In [44]:
def get_season(m):
    if m in [12, 1, 2]:   return 'Winter'
    elif m in [3, 4, 5]:  return 'Summer'
    elif m in [6, 7, 8, 9]: return 'Monsoon'
    else:                  return 'Post-Monsoon'

df['Season'] = df['Month'].apply(get_season)

In [45]:
case_cap = df['Cases'].quantile(0.99)
print(f"Capping Cases at 99th percentile: {case_cap:.2f}")
df['Cases'] = df['Cases'].clip(upper=case_cap)

deaths_cap = df['Deaths'].quantile(0.99)
df['Deaths'] = df['Deaths'].clip(upper=deaths_cap)

Capping Cases at 99th percentile: 836.56


In [46]:
df = df.sort_values(['state_ut', 'district', 'year_month']).reset_index(drop=True)

In [47]:
df['Cases_MA_3'] = df.groupby('district')['Cases'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['Deaths_MA_3'] = df.groupby('district')['Deaths'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

In [48]:
df['Case_Growth_Rate'] = df.groupby('district')['Cases'].transform(
    lambda x: x.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0).clip(-1, 5)
)

In [49]:
df = df.groupby(
    ['state_ut', 'district', 'year_month', 'Latitude', 'Longitude', 'Month', 'Season'],
    as_index=False
).agg({
    'Cases':            'sum',
    'Deaths':           'sum',
    'preci':            'mean',
    'LAI':              'mean',
    'Temp':             'mean',
    'Cases_MA_3':       'mean',
    'Deaths_MA_3':      'mean',
    'Case_Growth_Rate': 'mean'
})

In [51]:
print("Final shape:", df.shape)
print("\nMissing values:\n",  df.isnull().sum())


Final shape: (6709, 15)

Missing values:
 state_ut            0
district            0
year_month          0
Latitude            0
Longitude           0
Month               0
Season              0
Cases               0
Deaths              0
preci               0
LAI                 0
Temp                0
Cases_MA_3          0
Deaths_MA_3         0
Case_Growth_Rate    0
dtype: int64


In [53]:
print("\nInf values:\n",      np.isinf(df.select_dtypes(include=np.number)).sum())



Inf values:
 Latitude            0
Longitude           0
Month               0
Cases               0
Deaths              0
preci               0
LAI                 0
Temp                0
Cases_MA_3          0
Deaths_MA_3         0
Case_Growth_Rate    0
dtype: int64


In [54]:
print("\nDescriptive stats:\n", df[['Cases','Cases_MA_3','Deaths_MA_3','Case_Growth_Rate']].describe())
print("\nSample — Anantapur:")
print(df[df['district']=='Anantapur'][
    ['year_month','Cases','Cases_MA_3','Case_Growth_Rate','Season']
].to_string())


Descriptive stats:
              Cases   Cases_MA_3  Deaths_MA_3  Case_Growth_Rate
count  6709.000000  6709.000000  6709.000000       6709.000000
mean     79.976956    56.360362     0.312243          0.505805
std     225.279388    80.199110     0.631721          1.416708
min       0.000000     0.000000     0.000000         -0.996960
25%      17.000000    21.000000     0.000000         -0.368421
50%      33.000000    33.000000     0.000000          0.000000
75%      70.000000    57.000000     0.333333          0.839912
max    7923.480000   836.560000     5.000000          5.000000

Sample — Anantapur:
   year_month  Cases  Cases_MA_3  Case_Growth_Rate   Season
3  2010-06-01   68.0   68.000000          0.000000  Monsoon
4  2011-06-01  128.0   42.055556          0.634484  Monsoon
5  2011-09-01   25.0   43.333333         -0.695122  Monsoon
6  2012-05-01   25.0   44.000000          0.000000   Summer
7  2012-12-01   23.0   24.333333         -0.080000   Winter
8  2013-06-01   59.0   35.66666

In [55]:
df.to_csv("../data/processed/epiclim_preprocessed.csv", index=False)
print("\nSaved to ../data/processed/epiclim_preprocessed.csv")


Saved to ../data/processed/epiclim_preprocessed.csv


In [56]:
df[['Cases','Cases_MA_3']].describe()

,Cases,Cases_MA_3
count,6709.000000,6709.000000
mean,79.976956,56.360362
std,225.279388,80.199110
min,0.000000,0.000000
25%,17.000000,21.000000
50%,33.000000,33.000000
75%,70.000000,57.000000
max,7923.480000,836.560000
